# Python `bisect` ：二分搜尋與插入位置


1. `bisect_left()` / `bisect_right()` 是什麼
2. 如何在排序好的 list 裡找插入位置
3. 如何找最接近的數
4. 如何用 `bisect` 解 AtCoder ABC077 C / ARC084 A 類型題目
5. 為什麼 `bisect` 可以把暴力搜尋變快

## 0. 測速工具

In [2]:
from time import perf_counter

def show_time(title, func):
    start = perf_counter()
    result = func()
    end = perf_counter()
    print(title)
    print("結果:", result)
    print(f"時間: {end - start:.6f} 秒")
    return result


## 1. `bisect` 是什麼？

`bisect` 可以在「已排序」的 list 裡，用二分搜尋找到某個值應該插入的位置。

例如：

```python
a = [1, 3, 5, 7, 9]
x = 6
```

`6` 如果要插進去，應該插在 `5` 和 `7` 中間：

```text
[1, 3, 5, 6, 7, 9]
          ^
          index = 3
```


In [3]:
import bisect

a = [1, 3, 5, 7, 9]
x = 6

pos = bisect.bisect_left(a, x)

print("a =", a)
print("x =", x)
print("x 應該插入的位置 index =", pos)
print("插入後會變成：", a[:pos] + [x] + a[pos:])


a = [1, 3, 5, 7, 9]
x = 6
x 應該插入的位置 index = 3
插入後會變成： [1, 3, 5, 6, 7, 9]


## 2. `bisect_left()` 與 `bisect_right()` 差在哪？

當 list 裡面已經有一樣的值時，差別才明顯。

```python
a = [1, 2, 2, 2, 3]
```

`bisect_left(a, 2)` 會回傳最左邊可以插入的位置。

`bisect_right(a, 2)` 會回傳最右邊可以插入的位置。


In [4]:
a = [1, 2, 2, 2, 3]
x = 2

left = bisect.bisect_left(a, x)
right = bisect.bisect_right(a, x)

print("a =", a)
print("x =", x)
print("bisect_left(a, 2)  =", left)
print("bisect_right(a, 2) =", right)

print()
print("小於 2 的數量 =", left)
print("小於等於 2 的數量 =", right)
print("等於 2 的數量 =", right - left)


a = [1, 2, 2, 2, 3]
x = 2
bisect_left(a, 2)  = 1
bisect_right(a, 2) = 4

小於 2 的數量 = 1
小於等於 2 的數量 = 4
等於 2 的數量 = 3


## 3. 常用口訣

假設 `a` 已排序：

```python
bisect_left(a, x)
```

代表：

> 第一個 `>= x` 的位置，也等於 list 裡面 `< x` 的數量。

```python
bisect_right(a, x)
```

代表：

> 第一個 `> x` 的位置，也等於 list 裡面 `<= x` 的數量。


In [5]:
a = [1, 2, 2, 2, 3, 5, 8]
x = 2

print("a =", a)
print("x =", x)
print("小於 x 的數量:", bisect.bisect_left(a, x))
print("小於等於 x 的數量:", bisect.bisect_right(a, x))
print("等於 x 的數量:", bisect.bisect_right(a, x) - bisect.bisect_left(a, x))


a = [1, 2, 2, 2, 3, 5, 8]
x = 2
小於 x 的數量: 1
小於等於 x 的數量: 4
等於 x 的數量: 3


## 4. 找最接近 `x` 的數

投影片中的例子：

```python
a = [1, 3, 5, 7, 9]
x = 6
```

最接近 `6` 的可能只會在插入位置左右兩邊：

```text
[1, 3, 5, 7, 9]
       ^  ^
       5  7
```

所以只要比較 `before` 和 `after`。


In [6]:
def find_closest(a, x):
    pos = bisect.bisect_left(a, x)

    # x 比全部數字都小
    if pos == 0:
        return a[0]

    # x 比全部數字都大
    if pos == len(a):
        return a[-1]

    before = a[pos - 1]
    after = a[pos]

    if after - x < x - before:
        return after
    else:
        return before


a = [1, 3, 5, 7, 9]

for x in [0, 2, 4, 6, 8, 10]:
    print(f"x = {x}, 最接近的是 {find_closest(a, x)}")


x = 0, 最接近的是 1
x = 2, 最接近的是 1
x = 4, 最接近的是 3
x = 6, 最接近的是 5
x = 8, 最接近的是 7
x = 10, 最接近的是 9


## 5. 為什麼 `bisect` 快？

`in list` 或一個一個找通常是線性搜尋：

```text
O(n)
```

`bisect` 使用二分搜尋：

```text
O(log n)
```

但是注意：`bisect.insort(a, x)` 雖然找位置是 `O(log n)`，但插入 list 中間要搬動元素，所以整體仍可能是 `O(n)`。

本章主要用 `bisect` 做「查詢位置」，不是大量插入。


In [7]:
large = list(range(0, 5_000_000, 2))  # 偶數，已排序
target = 4_999_998

def linear_search():
    return target in large

def bisect_search():
    pos = bisect.bisect_left(large, target)
    return pos < len(large) and large[pos] == target

show_time("線性搜尋：target in large", linear_search)
show_time("bisect 二分搜尋", bisect_search)


線性搜尋：target in large
結果: True
時間: 0.023035 秒
bisect 二分搜尋
結果: True
時間: 0.000007 秒


True

## 6. AtCoder ABC077 C / ARC084 A 題目概念

題目有三個長度都是 `N` 的整數序列 `A`、`B`、`C`。

要算出有多少組：

```text
a < b < c
```

其中：

- `a` 來自 `A`
- `b` 來自 `B`
- `c` 來自 `C`

範例輸入：

```text
2
1 5
2 4
3 6
```

範例輸出：

```text
3
```


## 7. 暴力解法：三層迴圈

最直覺的寫法是：

```python
for a in A:
    for b in B:
        for c in C:
            if a < b < c:
                count += 1
```

時間複雜度是：

```text
O(N^3)
```

如果 `N = 100000`，一定跑不完。


In [8]:
def count_triplets_bruteforce(A, B, C):
    total = 0
    for a in A:
        for b in B:
            for c in C:
                if a < b < c:
                    total += 1
    return total


A = [1, 5]
B = [2, 4]
C = [3, 6]

print(count_triplets_bruteforce(A, B, C))


3


## 8. 用 `bisect` 改善想法

固定一個 `b`，我們要知道：

1. `A` 裡面有幾個數 `< b`
2. `C` 裡面有幾個數 `> b`

如果：

```text
A 中小於 b 的數量 = count_a
C 中大於 b 的數量 = count_c
```

那麼以這個 `b` 為中間值時，可以形成：

```text
count_a * count_c
```

組三元組。

使用 `bisect`：

```python
count_a = bisect_left(A, b)
```

因為 `bisect_left(A, b)` 等於 `A` 裡 `< b` 的數量。

```python
count_c = N - bisect_right(C, b)
```

因為 `bisect_right(C, b)` 等於 `C` 裡 `<= b` 的數量，所以右邊剩下的是 `> b` 的數量。


In [9]:
def count_triplets_bisect(N, A, B, C):
    A.sort()
    B.sort()
    C.sort()

    total = 0

    for b in B:
        count_a = bisect.bisect_left(A, b)
        count_c = N - bisect.bisect_right(C, b)
        total += count_a * count_c

    return total


N = 2
A = [1, 5]
B = [2, 4]
C = [3, 6]

print(count_triplets_bisect(N, A, B, C))


3


## 9. 過程展示

用範例：

```python
A = [1, 5]
B = [2, 4]
C = [3, 6]
```

### 當 b = 2

`A` 裡 `< 2` 的數有 `1`，所以 `count_a = 1`。

`C` 裡 `> 2` 的數有 `3, 6`，所以 `count_c = 2`。

貢獻：

```text
1 * 2 = 2
```

### 當 b = 4

`A` 裡 `< 4` 的數有 `1`，所以 `count_a = 1`。

`C` 裡 `> 4` 的數有 `6`，所以 `count_c = 1`。

貢獻：

```text
1 * 1 = 1
```

總答案：

```text
2 + 1 = 3
```


In [10]:
A = [1, 5]
B = [2, 4]
C = [3, 6]
N = 2

A.sort()
B.sort()
C.sort()

total = 0

for b in B:
    count_a = bisect.bisect_left(A, b)
    count_c = N - bisect.bisect_right(C, b)
    add = count_a * count_c
    total += add

    print("b =", b)
    print("A 中小於 b 的數量:", count_a)
    print("C 中大於 b 的數量:", count_c)
    print("這個 b 貢獻:", add)
    print()

print("答案:", total)


b = 2
A 中小於 b 的數量: 1
C 中大於 b 的數量: 2
這個 b 貢獻: 2

b = 4
A 中小於 b 的數量: 1
C 中大於 b 的數量: 1
這個 b 貢獻: 1

答案: 3


## 10. 比賽提交版

標準輸入，使用 `< sample_abc077c.txt` 測試。


In [11]:
%%writefile abc077c_bisect_solution.py
import bisect

def count_triplets(N, A, B, C):
    A.sort()
    B.sort()
    C.sort()

    total = 0

    for b in B:
        count_a = bisect.bisect_left(A, b)
        count_c = N - bisect.bisect_right(C, b)
        total += count_a * count_c

    return total


N = int(input())
A = list(map(int, input().split()))
B = list(map(int, input().split()))
C = list(map(int, input().split()))

print(count_triplets(N, A, B, C))


Writing abc077c_bisect_solution.py


In [12]:
%%writefile sample_abc077c.txt
2
1 5
2 4
3 6


Writing sample_abc077c.txt


In [13]:
!python abc077c_bisect_solution.py < sample_abc077c.txt

3


## 11. 與暴力法比較速度

暴力法是 `O(N^3)`。

`bisect` 解法：

1. 排序 `A, B, C`：`O(N log N)`
2. 對每個 `b` 做兩次二分搜尋：`O(N log N)`

總體：

```text
O(N log N)
```


In [14]:
import random

N_small = 150
A_small = [random.randint(1, 10_000) for _ in range(N_small)]
B_small = [random.randint(1, 10_000) for _ in range(N_small)]
C_small = [random.randint(1, 10_000) for _ in range(N_small)]

show_time("暴力法 O(N^3)", lambda: count_triplets_bruteforce(A_small, B_small, C_small))
show_time("bisect 法 O(N log N)", lambda: count_triplets_bisect(N_small, A_small[:], B_small[:], C_small[:]))


暴力法 O(N^3)
結果: 635250
時間: 0.079215 秒
bisect 法 O(N log N)
結果: 635250
時間: 0.000062 秒


635250

## 12. 常見錯誤

### 錯誤 1：忘記排序

`bisect` 只能用在排序好的 list。

```python
A.sort()
C.sort()
```

一定要記得。

### 錯誤 2：把 `< b` 和 `<= b` 搞混

這題要的是：

```text
a < b < c
```

所以：

```python
count_a = bisect_left(A, b)          # a < b
count_c = N - bisect_right(C, b)     # c > b
```

### 錯誤 3：看到 `bisect` 就以為一定要插入

比賽常用 `bisect` 是為了「算位置」或「算數量」，不一定是真的插入。


## 13. 課堂練習 1：算小於 x 的數量

給定已排序 list：

```python
a = [1, 2, 2, 4, 7, 10]
```

請寫程式算出：

1. 小於 `4` 的數量
2. 小於等於 `4` 的數量
3. 等於 `2` 的數量


In [15]:
# 參考答案

a = [1, 2, 2, 4, 7, 10]

print("小於 4 的數量:", bisect.bisect_left(a, 4))
print("小於等於 4 的數量:", bisect.bisect_right(a, 4))
print("等於 2 的數量:", bisect.bisect_right(a, 2) - bisect.bisect_left(a, 2))


小於 4 的數量: 3
小於等於 4 的數量: 4
等於 2 的數量: 2


## 14. 課堂練習 2：找最接近的分數

給定排序好的分數：

```python
scores = [60, 70, 75, 82, 90, 100]
```

請找出最接近 `x = 78` 的分數。


In [16]:
# 參考答案

scores = [60, 70, 75, 82, 90, 100]
x = 78

print(find_closest(scores, x))


75


## 15. 課堂練習 3：ABC077 C 小測試

請自己修改下面的 `A, B, C`，確認答案是否合理。


In [17]:
N = 3
A = [1, 2, 3]
B = [2, 3, 4]
C = [3, 4, 5]

print(count_triplets_bisect(N, A, B, C))


10


# 總結

`bisect` 最常記這三件事：

```python
bisect_left(a, x)
```

代表：

```text
第一個 >= x 的位置
也就是小於 x 的數量
```

```python
bisect_right(a, x)
```

代表：

```text
第一個 > x 的位置
也就是小於等於 x 的數量
```

使用前一定要：

```python
a.sort()
```

競賽常用：

```python
count_less = bisect.bisect_left(a, x)
count_greater = len(a) - bisect.bisect_right(a, x)
```
